## CIS 508 Final Project

### Importing Libraries

In [1]:
# Core
import pandas as pd
import numpy as np
from pathlib import Path

# Preprocessing
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Model evaluation
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

# Regression models
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# Neural network regressor
from sklearn.neural_network import MLPRegressor

# XGBoost
from xgboost import XGBRegressor

# Model saving
import joblib

# Plotting (for residual plots / confusion-matrix style visuals)
import matplotlib.pyplot as plt
import seaborn as sns

# Misc
import warnings
warnings.filterwarnings("ignore")

### Data Inspection

#### Importing Data

In [2]:
DATA_PATH = Path("../data/rideshare_kaggle.csv")

df = pd.read_csv(DATA_PATH)

print("✔️ Data loaded successfully!")

✔️ Data loaded successfully!


#### Basic Inspection

In [3]:
# Shape of dataset
print("Shape:", df.shape)

# First 5 rows
display(df.head())

# Data types + non-null info
print("\nInfo:")
df.info()

# Summary statistics (numerical)
print("\nDescribe (numerical):")
display(df.describe())

# Summary statistics (categorical)
print("\nDescribe (categorical):")
display(df.describe(include='object'))

Shape: (693071, 57)


,id,timestamp,hour,day,month,datetime,timezone,source,destination,cab_type,...,precipIntensityMax,uvIndexTime,temperatureMin,temperatureMinTime,temperatureMax,temperatureMaxTime,apparentTemperatureMin,apparentTemperatureMinTime,apparentTemperatureMax,apparentTemperatureMaxTime
0,424553bb-7174-41ea-aeb4-fe06d4f4b9d7,1.544953e+09,9,16,12,2018-12-16 09:30:07,America/New_York,Haymarket Square,North Station,Lyft,...,0.1276,1544979600,39.89,1545012000,43.68,1544968800,33.73,1545012000,38.07,1544958000
1,4bd23055-6827-41c6-b23b-3c491f24e74d,1.543284e+09,2,27,11,2018-11-27 02:00:23,America/New_York,Haymarket Square,North Station,Lyft,...,0.1300,1543251600,40.49,1543233600,47.30,1543251600,36.20,1543291200,43.92,1543251600
2,981a3613-77af-4620-a42a-0c0866077d1e,1.543367e+09,1,28,11,2018-11-28 01:00:22,America/New_York,Haymarket Square,North Station,Lyft,...,0.1064,1543338000,35.36,1543377600,47.55,1543320000,31.04,1543377600,44.12,1543320000
3,c2d88af2-d278-4bfd-a8d0-29ca77cc5512,1.543554e+09,4,30,11,2018-11-30 04:53:02,America/New_York,Haymarket Square,North Station,Lyft,...,0.0000,1543507200,34.67,1543550400,45.03,1543510800,30.30,1543550400,38.53,1543510800
4,e0126e1f-8ca9-4f2e-82b3-50505a09db9a,1.543463e+09,3,29,11,2018-11-29 03:49:20,America/New_York,Haymarket Square,North Station,Lyft,...,0.0001,1543420800,33.10,1543402800,42.18,1543420800,29.11,1543392000,35.75,1543420800



Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 693071 entries, 0 to 693070
Data columns (total 57 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           693071 non-null  object 
 1   timestamp                    693071 non-null  float64
 2   hour                         693071 non-null  int64  
 3   day                          693071 non-null  int64  
 4   month                        693071 non-null  int64  
 5   datetime                     693071 non-null  object 
 6   timezone                     693071 non-null  object 
 7   source                       693071 non-null  object 
 8   destination                  693071 non-null  object 
 9   cab_type                     693071 non-null  object 
 10  product_id                   693071 non-null  object 
 11  name                         693071 non-null  object 
 12  price                        637976 non-null  float

,timestamp,hour,day,month,price,distance,surge_multiplier,latitude,longitude,temperature,...,precipIntensityMax,uvIndexTime,temperatureMin,temperatureMinTime,temperatureMax,temperatureMaxTime,apparentTemperatureMin,apparentTemperatureMinTime,apparentTemperatureMax,apparentTemperatureMaxTime
count,6.930710e+05,693071.000000,693071.000000,693071.000000,637976.000000,693071.000000,693071.000000,693071.000000,693071.000000,693071.000000,...,693071.000000,6.930710e+05,693071.000000,6.930710e+05,693071.000000,6.930710e+05,693071.000000,6.930710e+05,693071.000000,6.930710e+05
mean,1.544046e+09,11.619137,17.794365,11.586684,16.545125,2.189430,1.013870,42.338172,-71.066151,39.584388,...,0.037374,1.544044e+09,33.457774,1.544042e+09,45.261313,1.544047e+09,29.731002,1.544048e+09,41.997343,1.544048e+09
std,6.891925e+05,6.948114,9.982286,0.492429,9.324359,1.138937,0.091641,0.047840,0.020302,6.726084,...,0.055214,6.912028e+05,6.467224,6.901954e+05,5.645046,6.901353e+05,7.110494,6.871862e+05,6.936841,6.910777e+05
min,1.543204e+09,0.000000,1.000000,11.000000,2.500000,0.020000,1.000000,42.214800,-71.105400,18.910000,...,0.000000,1.543162e+09,15.630000,1.543122e+09,33.510000,1.543154e+09,11.810000,1.543136e+09,28.950000,1.543187e+09
25%,1.543444e+09,6.000000,13.000000,11.000000,9.000000,1.280000,1.000000,42.350300,-71.081000,36.450000,...,0.000000,1.543421e+09,30.170000,1.543399e+09,42.570000,1.543439e+09,27.760000,1.543399e+09,36.570000,1.543439e+09
50%,1.543737e+09,12.000000,17.000000,12.000000,13.500000,2.160000,1.000000,42.351900,-71.063100,40.490000,...,0.000400,1.543770e+09,34.240000,1.543727e+09,44.680000,1.543788e+09,30.130000,1.543745e+09,40.950000,1.543788e+09
75%,1.544828e+09,18.000000,28.000000,12.000000,22.500000,2.920000,1.000000,42.364700,-71.054200,43.580000,...,0.091600,1.544807e+09,38.880000,1.544789e+09,46.910000,1.544814e+09,35.710000,1.544789e+09,44.120000,1.544818e+09
max,1.545161e+09,23.000000,30.000000,12.000000,97.500000,7.860000,3.000000,42.366100,-71.033000,57.220000,...,0.145900,1.545152e+09,43.100000,1.545192e+09,57.870000,1.545109e+09,40.050000,1.545134e+09,57.200000,1.545109e+09



Describe (categorical):


,id,datetime,timezone,source,destination,cab_type,product_id,name,short_summary,long_summary,icon
count,693071,693071,693071,693071,693071,693071,693071,693071,693071,693071,693071
unique,693071,31350,1,12,12,2,13,13,9,11,7
top,424553bb-7174-41ea-aeb4-fe06d4f4b9d7,2018-11-26 23:21:14,America/New_York,Financial District,Financial District,Uber,6f72dfc5-27f1-42e8-84db-ccc7a75f6969,UberXL,Overcast,Mostly cloudy throughout the day.,cloudy
freq,1,156,693071,58857,58851,385663,55096,55096,218895,202340,218895


#### Missing Values

In [4]:
# Count missing values per column
null_counts = df.isnull().sum().sort_values(ascending=False)

print("\nMissing Values Per Column:")
display(null_counts)

# Total rows with at least one missing value
total_null_rows = df.isnull().any(axis=1).sum()
print("\nTotal rows with at least one null:", total_null_rows)


Missing Values Per Column:


price                          55095
id                                 0
ozone                              0
temperatureLowTime                 0
apparentTemperatureHigh            0
apparentTemperatureHighTime        0
apparentTemperatureLow             0
apparentTemperatureLowTime         0
icon                               0
dewPoint                           0
pressure                           0
windBearing                        0
cloudCover                         0
uvIndex                            0
visibility.1                       0
sunriseTime                        0
temperatureHighTime                0
sunsetTime                         0
moonPhase                          0
precipIntensityMax                 0
uvIndexTime                        0
temperatureMin                     0
temperatureMinTime                 0
temperatureMax                     0
temperatureMaxTime                 0
apparentTemperatureMin             0
apparentTemperatureMinTime         0
a


Total rows with at least one null: 55095


#### Duplicate Check

In [5]:
duplicate_count = df.duplicated().sum()
print("\nDuplicate rows:", duplicate_count)


Duplicate rows: 0


#### Columns Check

In [6]:
print("\nColumn Names:")
print(list(df.columns))


Column Names:
['id', 'timestamp', 'hour', 'day', 'month', 'datetime', 'timezone', 'source', 'destination', 'cab_type', 'product_id', 'name', 'price', 'distance', 'surge_multiplier', 'latitude', 'longitude', 'temperature', 'apparentTemperature', 'short_summary', 'long_summary', 'precipIntensity', 'precipProbability', 'humidity', 'windSpeed', 'windGust', 'windGustTime', 'visibility', 'temperatureHigh', 'temperatureHighTime', 'temperatureLow', 'temperatureLowTime', 'apparentTemperatureHigh', 'apparentTemperatureHighTime', 'apparentTemperatureLow', 'apparentTemperatureLowTime', 'icon', 'dewPoint', 'pressure', 'windBearing', 'cloudCover', 'uvIndex', 'visibility.1', 'ozone', 'sunriseTime', 'sunsetTime', 'moonPhase', 'precipIntensityMax', 'uvIndexTime', 'temperatureMin', 'temperatureMinTime', 'temperatureMax', 'temperatureMaxTime', 'apparentTemperatureMin', 'apparentTemperatureMinTime', 'apparentTemperatureMax', 'apparentTemperatureMaxTime']


#### Distribution Check for Key Fields

In [7]:
# Useful sanity checks
print("\nPrice distribution:")
display(df["price"].describe())

print("\nDistance distribution:")
display(df["distance"].describe())

print("\nSurge multiplier distribution:")
display(df["surge_multiplier"].value_counts().sort_index())


Price distribution:


count    637976.000000
mean         16.545125
std           9.324359
min           2.500000
25%           9.000000
50%          13.500000
75%          22.500000
max          97.500000
Name: price, dtype: float64


Distance distribution:


count    693071.000000
mean          2.189430
std           1.138937
min           0.020000
25%           1.280000
50%           2.160000
75%           2.920000
max           7.860000
Name: distance, dtype: float64


Surge multiplier distribution:


surge_multiplier
1.00    672096
1.25     11085
1.50      5065
1.75      2420
2.00      2239
2.50       154
3.00        12
Name: count, dtype: int64

#### DateTime Check

In [8]:
print("\nDatetime field ready?:", df["datetime"].dtype)


Datetime field ready?: object


### Pre-Processing

#### Removing Unneccessary Columns

In [9]:
cols_to_drop = [
    "id",
    "timestamp",
    "timezone",
    "product_id",
    "latitude",
    "longitude",
    "long_summary",
    "windGustTime",
    "temperatureHighTime",
    "temperatureLowTime",
    "apparentTemperatureHighTime",
    "apparentTemperatureLowTime",
    "icon",
    "visibility.1",
    "sunriseTime",
    "sunsetTime",
    "uvIndexTime",
    "temperatureMinTime",
    "temperatureMaxTime",
    "apparentTemperatureMinTime",
    "apparentTemperatureMaxTime",
]

df_reduced = df.drop(columns=cols_to_drop)

print("Shape before dropping:", df.shape)
print("Shape after dropping:", df_reduced.shape)

# Check remaining columns
df_reduced.head()

Shape before dropping: (693071, 57)
Shape after dropping: (693071, 36)


,hour,day,month,datetime,source,destination,cab_type,name,price,distance,...,windBearing,cloudCover,uvIndex,ozone,moonPhase,precipIntensityMax,temperatureMin,temperatureMax,apparentTemperatureMin,apparentTemperatureMax
0,9,16,12,2018-12-16 09:30:07,Haymarket Square,North Station,Lyft,Shared,5.0,0.44,...,57,0.72,0,303.8,0.30,0.1276,39.89,43.68,33.73,38.07
1,2,27,11,2018-11-27 02:00:23,Haymarket Square,North Station,Lyft,Lux,11.0,0.44,...,90,1.00,0,291.1,0.64,0.1300,40.49,47.30,36.20,43.92
2,1,28,11,2018-11-28 01:00:22,Haymarket Square,North Station,Lyft,Lyft,7.0,0.44,...,240,0.03,0,315.7,0.68,0.1064,35.36,47.55,31.04,44.12
3,4,30,11,2018-11-30 04:53:02,Haymarket Square,North Station,Lyft,Lux Black XL,26.0,0.44,...,310,0.00,0,291.1,0.75,0.0000,34.67,45.03,30.30,38.53
4,3,29,11,2018-11-29 03:49:20,Haymarket Square,North Station,Lyft,Lyft XL,9.0,0.44,...,303,0.44,0,347.7,0.72,0.0001,33.10,42.18,29.11,35.75


#### Removing Null Values

In [10]:
# Drop all rows containing at least one null
df_clean = df_reduced.dropna().copy()

print("Shape before dropping:", df_reduced.shape)
print("Shape after dropping ALL nulls:", df_clean.shape)

# Confirm the dataset has no nulls left
df_clean.isnull().sum().sum()

Shape before dropping: (693071, 36)
Shape after dropping ALL nulls: (637976, 36)


np.int64(0)

### Feature Engineering

In [11]:
# 1. Parse datetime
df_clean["datetime"] = pd.to_datetime(df_clean["datetime"])

# 2. Convert month number → month name
df_clean["month_name"] = df_clean["datetime"].dt.month_name()

# 3. Add weekday names (Monday, Tuesday, ...)
df_clean["day_name"] = df_clean["datetime"].dt.day_name()

# 4. Weekend flag
df_clean["is_weekend"] = df_clean["day_name"].isin(["Saturday", "Sunday"]).astype(int)

# Preview
print(df_clean[["datetime", "hour", "month", "month_name", "day_name", "is_weekend"]].head())

# 5. Drop raw datetime column
df_clean = df_clean.drop(columns=["datetime"])

# 6. Drop numeric day-of-month
df_clean = df_clean.drop(columns=["day"])

# 7. Drop numeric month column (we replaced it with month_name)
df_clean = df_clean.drop(columns=["month"])

df_clean.head()

             datetime  hour  month month_name   day_name  is_weekend
0 2018-12-16 09:30:07     9     12   December     Sunday           1
1 2018-11-27 02:00:23     2     11   November    Tuesday           0
2 2018-11-28 01:00:22     1     11   November  Wednesday           0
3 2018-11-30 04:53:02     4     11   November     Friday           0
4 2018-11-29 03:49:20     3     11   November   Thursday           0


,hour,source,destination,cab_type,name,price,distance,surge_multiplier,temperature,apparentTemperature,...,ozone,moonPhase,precipIntensityMax,temperatureMin,temperatureMax,apparentTemperatureMin,apparentTemperatureMax,month_name,day_name,is_weekend
0,9,Haymarket Square,North Station,Lyft,Shared,5.0,0.44,1.0,42.34,37.12,...,303.8,0.30,0.1276,39.89,43.68,33.73,38.07,December,Sunday,1
1,2,Haymarket Square,North Station,Lyft,Lux,11.0,0.44,1.0,43.58,37.35,...,291.1,0.64,0.1300,40.49,47.30,36.20,43.92,November,Tuesday,0
2,1,Haymarket Square,North Station,Lyft,Lyft,7.0,0.44,1.0,38.33,32.93,...,315.7,0.68,0.1064,35.36,47.55,31.04,44.12,November,Wednesday,0
3,4,Haymarket Square,North Station,Lyft,Lux Black XL,26.0,0.44,1.0,34.38,29.63,...,291.1,0.75,0.0000,34.67,45.03,30.30,38.53,November,Friday,0
4,3,Haymarket Square,North Station,Lyft,Lyft XL,9.0,0.44,1.0,37.44,30.88,...,347.7,0.72,0.0001,33.10,42.18,29.11,35.75,November,Thursday,0


### VIF Analysis

In [12]:
df_vif = df_clean.copy()  # start from your cleaned dataframe

# Select numeric columns and exclude target
numeric_cols = df_vif.select_dtypes(include=["float64", "int64"]).columns.tolist()
if "price" in numeric_cols:
    numeric_cols.remove("price")

print("Initial numeric columns for VIF:")
print(numeric_cols)

vif_threshold = 10.0
dropped_features = []

def compute_vif(df, features):
    """Compute VIF for a list of numeric features."""
    X = df[features].values
    vif_data = []
    for i, col in enumerate(features):
        vif_val = variance_inflation_factor(X, i)
        vif_data.append({"feature": col, "vif": vif_val})
    vif_df = (
        pd.DataFrame(vif_data)
        .sort_values("vif", ascending=False)
        .reset_index(drop=True)
    )
    return vif_df

iteration = 1

# ---- VIF LOOP ----
while True:
    print(f"\nIteration {iteration}")
    vif_df = compute_vif(df_vif, numeric_cols)
    display(vif_df)

    max_vif = vif_df["vif"].max()
    if max_vif < vif_threshold:
        print(f"All remaining features have VIF < {vif_threshold}. Stopping VIF elimination loop.")
        break

    # feature with highest VIF
    row_max = vif_df.loc[vif_df["vif"].idxmax()]
    feature_to_drop = row_max["feature"]
    feature_vif = row_max["vif"]

    print(f"Dropping feature: {feature_to_drop} with VIF = {feature_vif:.4f}")

    numeric_cols.remove(feature_to_drop)
    dropped_features.append((feature_to_drop, feature_vif))
    df_vif = df_vif.drop(columns=[feature_to_drop])

    iteration += 1

# ---- MANUALLY RE-ADD SURGE_MULTIPLIER ----
print("\nRe-adding surge_multiplier (important business feature)")

df_vif["surge_multiplier"] = df_clean["surge_multiplier"]

if "surge_multiplier" not in numeric_cols:
    numeric_cols.append("surge_multiplier")

print("\nFinal numeric features after VIF filtering + surge_multiplier restored:")
print(numeric_cols)

# ---- FINAL VIF TABLE ----
print("\nRecomputing VIF including surge_multiplier:")
final_vif_df = compute_vif(df_vif, numeric_cols)
display(final_vif_df)

# ---- REPORT DROPPED FEATURES ----
print("\nDropped features due to high VIF:")
for feat, v in dropped_features:
    print(f"• {feat}: VIF = {v:.4f}")

Initial numeric columns for VIF:
['hour', 'distance', 'surge_multiplier', 'temperature', 'apparentTemperature', 'precipIntensity', 'precipProbability', 'humidity', 'windSpeed', 'windGust', 'visibility', 'temperatureHigh', 'temperatureLow', 'apparentTemperatureHigh', 'apparentTemperatureLow', 'dewPoint', 'pressure', 'windBearing', 'cloudCover', 'uvIndex', 'ozone', 'moonPhase', 'precipIntensityMax', 'temperatureMin', 'temperatureMax', 'apparentTemperatureMin', 'apparentTemperatureMax', 'is_weekend']

Iteration 1


,feature,vif
0,temperatureHigh,2.179874e+06
1,temperatureMax,2.174095e+06
2,apparentTemperatureHigh,6.370665e+05
3,apparentTemperatureMax,6.370149e+05
4,temperature,2.137709e+04
5,dewPoint,1.154991e+04
6,pressure,1.037005e+04
7,humidity,7.513347e+03
8,apparentTemperature,2.357615e+03
9,apparentTemperatureMin,1.722819e+03


Dropping feature: temperatureHigh with VIF = 2179873.6652

Iteration 2


,feature,vif
0,temperature,21023.785384
1,dewPoint,11279.574535
2,pressure,10251.015746
3,humidity,7377.905659
4,apparentTemperatureMax,5124.432913
5,temperatureMax,4312.357121
6,apparentTemperatureHigh,2820.017816
7,apparentTemperature,2354.962965
8,apparentTemperatureMin,1683.905615
9,temperatureMin,1397.839358


Dropping feature: temperature with VIF = 21023.7854

Iteration 3


...

Dropping feature: visibility with VIF = 14.2011

Iteration 18


,feature,vif
0,windBearing,9.520698
1,moonPhase,7.652852
2,cloudCover,5.436620
3,precipProbability,4.855939
4,windGust,4.446128
5,hour,4.256464
6,precipIntensity,4.230362
7,distance,4.025936
8,precipIntensityMax,2.805840
9,uvIndex,1.462746


All remaining features have VIF < 10.0. Stopping VIF elimination loop.

Re-adding surge_multiplier (important business feature)

Final numeric features after VIF filtering + surge_multiplier restored:
['hour', 'distance', 'precipIntensity', 'precipProbability', 'windGust', 'windBearing', 'cloudCover', 'uvIndex', 'moonPhase', 'precipIntensityMax', 'is_weekend', 'surge_multiplier']

Recomputing VIF including surge_multiplier:


,feature,vif
0,surge_multiplier,21.234022
1,windBearing,12.253425
2,moonPhase,7.947330
3,cloudCover,5.828114
4,precipProbability,4.859848
5,hour,4.635474
6,distance,4.626445
7,windGust,4.537849
8,precipIntensity,4.289132
9,precipIntensityMax,2.886936



Dropped features due to high VIF:
• temperatureHigh: VIF = 2179873.6652
• temperature: VIF = 21023.7854
• apparentTemperatureMax: VIF = 5100.2017
• pressure: VIF = 2291.0092
• temperatureMax: VIF = 1529.5072
• temperatureMin: VIF = 1089.5088
• temperatureLow: VIF = 757.1757
• dewPoint: VIF = 748.3663
• apparentTemperatureHigh: VIF = 179.0202
• ozone: VIF = 170.3511
• apparentTemperatureMin: VIF = 93.4112
• surge_multiplier: VIF = 77.0750
• windSpeed: VIF = 56.2624
• humidity: VIF = 50.1542
• apparentTemperature: VIF = 30.3195
• apparentTemperatureLow: VIF = 21.8781
• visibility: VIF = 14.2011
